In [1]:
# Cell 1: Build BeH2 fermionic Hamiltonian and Hermitian fermionic terms

import math
import pandas as pd

from openfermion.chem import MolecularData
from openfermion.ops import FermionOperator
from openfermion.transforms import get_fermion_operator, normal_ordered
from openfermion.utils import hermitian_conjugated
from openfermionpyscf import run_pyscf

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Beryllium hydride / BeH2 configuration
# ------------------------------------------------------------

BEH2_BOND_LENGTH = 1.326  # Angstrom, approximate Be-H bond length
BASIS = "sto-3g"
MULTIPLICITY = 1
CHARGE = 0

# Full BeH2/STO-3G should give 14 qubits:
# Be has 5 STO-3G spatial orbitals and each H has 1.
# 7 spatial orbitals * 2 spin orbitals = 14 qubits.
USE_ACTIVE_SPACE = False

# Optional active-space example:
# Freeze the Be core-like spatial orbital and keep valence orbitals active.
# This usually reduces BeH2/STO-3G from 14 qubits to 12 qubits.
OCCUPIED_INDICES = [0]
ACTIVE_INDICES = [1, 2, 3, 4, 5, 6]

TERM_ABS_TOL = 1e-12
PRINT_FULL_HAMILTONIAN = True


def format_fermion_term(term):
    if term == ():
        return "I"

    pieces = []
    for orbital, action in term:
        if action == 1:
            pieces.append(f"a_{orbital}^dagger")
        else:
            pieces.append(f"a_{orbital}")
    return " ".join(pieces)


def sort_fermion_key(term):
    return (len(term), term)


def coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_fermion_key(item[0])):
        pieces.append(f"{coeff_to_str(coeff, digits)} {format_fermion_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def dagger_term_key(term):
    """
    Return the OpenFermion key for O^dagger, where O is one monomial.
    """
    O = FermionOperator(term, 1.0)
    O_dag = normal_ordered(hermitian_conjugated(O))
    O_dag.compress(abs_tol=TERM_ABS_TOL)

    if len(O_dag.terms) != 1:
        raise ValueError(f"Expected one dagger term, got: {O_dag}")

    return next(iter(O_dag.terms.keys()))


def make_hermitian_fermionic_terms(fermion_hamiltonian, tol=TERM_ABS_TOL):
    """
    Group raw monomials into Hermitian fermionic Hamiltonian terms.
    """
    used = set()
    hermitian_terms = []

    for term, coeff in fermion_hamiltonian.terms.items():
        if term in used:
            continue

        dag = dagger_term_key(term)

        if dag == term:
            T = FermionOperator(term, coeff)
            used.add(term)
        else:
            dag_coeff = fermion_hamiltonian.terms.get(dag, 0.0)

            T = FermionOperator(term, coeff)
            T += FermionOperator(dag, dag_coeff)

            used.add(term)
            used.add(dag)

        T = normal_ordered(T)
        T.compress(abs_tol=tol)
        hermitian_terms.append(T)

    return hermitian_terms


def infer_n_qubits_from_fermion_operator(op):
    """
    Infer the number of spin orbitals used by the FermionOperator.
    """
    max_orbital = -1

    for term in op.terms:
        for orbital, action in term:
            max_orbital = max(max_orbital, orbital)

    return max_orbital + 1


def build_beh2_geometry(beh2_bond_length=BEH2_BOND_LENGTH):
    """
    Build linear BeH2 geometry.

    Be is placed at the origin.
    The two H atoms are placed symmetrically on the z-axis.
    """
    geometry = [
        ("H",  (0.0, 0.0, -beh2_bond_length)),
        ("Be", (0.0, 0.0, 0.0)),
        ("H",  (0.0, 0.0, beh2_bond_length)),
    ]

    return geometry


def build_beh2_fermionic_hamiltonian(
    beh2_bond_length=BEH2_BOND_LENGTH,
    basis=BASIS,
    multiplicity=MULTIPLICITY,
    charge=CHARGE,
    use_active_space=USE_ACTIVE_SPACE,
    occupied_indices=OCCUPIED_INDICES,
    active_indices=ACTIVE_INDICES,
):
    """
    Build a BeH2 fermionic Hamiltonian using OpenFermion + PySCF.
    """
    geometry = build_beh2_geometry(beh2_bond_length=beh2_bond_length)

    molecule = MolecularData(
        geometry=geometry,
        basis=basis,
        multiplicity=multiplicity,
        charge=charge,
        description=f"BeH2_{beh2_bond_length}",
    )

    molecule = run_pyscf(
        molecule,
        run_scf=True,
        run_fci=False,
    )

    if use_active_space:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian(
            occupied_indices=occupied_indices,
            active_indices=active_indices,
        )
    else:
        molecular_hamiltonian = molecule.get_molecular_hamiltonian()

    fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
    fermion_hamiltonian = normal_ordered(fermion_hamiltonian)
    fermion_hamiltonian.compress(abs_tol=TERM_ABS_TOL)

    n_qubits = infer_n_qubits_from_fermion_operator(fermion_hamiltonian)

    return molecule, fermion_hamiltonian, n_qubits


# ------------------------------------------------------------
# Build BeH2 fermionic Hamiltonian
# ------------------------------------------------------------

molecule, Hf, n_qubits = build_beh2_fermionic_hamiltonian()
hermitian_terms = make_hermitian_fermionic_terms(Hf)

print("Molecule: BeH2 / Beryllium hydride")
print("Basis:", BASIS)
print("Be-H bond length [Angstrom]:", BEH2_BOND_LENGTH)
print("Use active space:", USE_ACTIVE_SPACE)
if USE_ACTIVE_SPACE:
    print("Frozen occupied spatial orbitals:", OCCUPIED_INDICES)
    print("Active spatial orbitals:", ACTIVE_INDICES)
print("Full molecule electrons:", molecule.n_electrons)
print("Full molecule spatial orbitals:", molecule.n_orbitals)
print("Full molecule spin orbitals / qubits:", molecule.n_qubits)
print("Hamiltonian spin orbitals / qubits used:", n_qubits)
print("Number of raw OpenFermion monomial terms:", len(Hf.terms))
print("Number of Hermitian fermionic terms:", len(hermitian_terms))

if PRINT_FULL_HAMILTONIAN:
    print("\n=== Full fermionic Hamiltonian H_f ===")
    print(Hf)


# ------------------------------------------------------------
# Tables
# ------------------------------------------------------------

raw_rows = []

for idx, (term, coeff) in enumerate(
    sorted(Hf.terms.items(), key=lambda item: sort_fermion_key(item[0]))
):
    raw_rows.append(
        {
            "raw_index": idx,
            "coefficient": coeff_to_str(coeff),
            "monomial": format_fermion_term(term),
            "OpenFermion_key": term,
        }
    )

raw_df = pd.DataFrame(raw_rows)

print("\n=== Raw fermionic monomials c_alpha O_alpha ===")
display(raw_df)


hermitian_rows = []

for i, T in enumerate(hermitian_terms):
    hermitian_rows.append(
        {
            "vertex": f"T_{i}",
            "number_of_monomials": len(T.terms),
            "fermionic_term": operator_to_string(T),
        }
    )

hermitian_df = pd.DataFrame(hermitian_rows)

print("\n=== Hermitian fermionic terms T_i ===")
display(hermitian_df)

Molecule: BeH2 / Beryllium hydride
Basis: sto-3g
Be-H bond length [Angstrom]: 1.326
Use active space: False
Full molecule electrons: 6
Full molecule spatial orbitals: 7
Full molecule spin orbitals / qubits: 14
Hamiltonian spin orbitals / qubits used: 14
Number of raw OpenFermion monomial terms: 666
Number of Hermitian fermionic terms: 386

=== Full fermionic Hamiltonian H_f ===
3.3921616084615387 [] +
-8.653614060526332 [0^ 0] +
0.2257881707079464 [0^ 2] +
0.1933759038468936 [0^ 10] +
-2.271489014068697 [1^ 0^ 1 0] +
-0.19912676459351364 [1^ 0^ 2 1] +
0.19912676459351364 [1^ 0^ 3 0] +
-0.026786606586168524 [1^ 0^ 3 2] +
-0.006047866793733274 [1^ 0^ 5 4] +
-0.01576724793895085 [1^ 0^ 7 6] +
-0.01576724793895087 [1^ 0^ 9 8] +
-0.1809163218063341 [1^ 0^ 10 1] +
0.02500722144397735 [1^ 0^ 10 3] +
0.1809163218063341 [1^ 0^ 11 0] +
-0.02500722144397735 [1^ 0^ 11 2] +
-0.02358722217677317 [1^ 0^ 11 10] +
0.011268108228017996 [1^ 0^ 12 5] +
-0.011268108228017996 [1^ 0^ 13 4] +
-0.0214323344268

,raw_index,coefficient,monomial,OpenFermion_key
0,0,+3.39216161,I,()
1,1,-8.65361406,a_0^dagger a_0,"((0, 1), (0, 0))"
2,2,+0.22578817,a_0^dagger a_2,"((0, 1), (2, 0))"
3,3,+0.19337590,a_0^dagger a_10,"((0, 1), (10, 0))"
4,4,-8.65361406,a_1^dagger a_1,"((1, 1), (1, 0))"
...,...,...,...,...
661,661,+0.05591151,a_13^dagger a_12^dagger a_11 a_2,"((13, 1), (12, 1), (11, 0), (2, 0))"
662,662,-0.14081677,a_13^dagger a_12^dagger a_11 a_10,"((13, 1), (12, 1), (11, 0), (10, 0))"
663,663,-0.01142776,a_13^dagger a_12^dagger a_12 a_5,"((13, 1), (12, 1), (12, 0), (5, 0))"
664,664,+0.01142776,a_13^dagger a_12^dagger a_13 a_4,"((13, 1), (12, 1), (13, 0), (4, 0))"



=== Hermitian fermionic terms T_i ===


,vertex,number_of_monomials,fermionic_term
0,T_0,1,+3.39216161 I
1,T_1,1,-8.65361406 a_0^dagger a_0
2,T_2,2,+0.22578817 a_0^dagger a_2 + +0.22578817 a_2^dagger a_0
3,T_3,2,+0.19337590 a_0^dagger a_10 + +0.19337590 a_10^dagger a_0
4,T_4,1,-8.65361406 a_1^dagger a_1
...,...,...,...
381,T_381,2,+0.14081677 a_12^dagger a_11^dagger a_13 a_10 + +0.14081677 a_13^dagger a_10^dagger a_12 a_11
382,T_382,1,-0.43649838 a_13^dagger a_10^dagger a_13 a_10
383,T_383,1,-0.43649838 a_12^dagger a_11^dagger a_12 a_11
384,T_384,1,-0.29568160 a_13^dagger a_11^dagger a_13 a_11


In [2]:
# Cell 2: Build the H2 fermionic noncommutation graph

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


def fermionic_commutator(A, B, tol=1e-12):
    """
    Compute [A, B] = AB - BA directly in the fermionic algebra.
    """
    C = normal_ordered(A * B - B * A)
    C.compress(abs_tol=tol)
    return C


def commute(A, B, tol=1e-12):
    """
    Return True if [A, B] = 0.
    """
    C = fermionic_commutator(A, B, tol=tol)
    return len(C.terms) == 0


# ------------------------------------------------------------
# Build noncommutation graph
# ------------------------------------------------------------
# Vertex i = Hermitian fermionic term T_i
# Edge (i, j) exists if [T_i, T_j] != 0

G = nx.Graph()

for i, T in enumerate(hermitian_terms):
    G.add_node(
        i,
        label=f"T_{i}",
        operator=T,
        operator_string=operator_to_string(T),
        number_of_monomials=len(T.terms),
    )

for i in range(len(hermitian_terms)):
    for j in range(i + 1, len(hermitian_terms)):
        Cij = fermionic_commutator(hermitian_terms[i], hermitian_terms[j])

        if len(Cij.terms) != 0:
            G.add_edge(
                i,
                j,
                commutator=Cij,
                commutator_string=operator_to_string(Cij),
            )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_vertices = G.number_of_nodes()
n_total_pairs = n_vertices * (n_vertices - 1) // 2
n_noncommuting_pairs = G.number_of_edges()
n_commuting_pairs = n_total_pairs - n_noncommuting_pairs

print("=== Fermionic noncommutation graph summary ===")
print("Number of vertices / fermionic terms:", n_vertices)
print("Number of total unordered pairs:", n_total_pairs)
print("Number of noncommuting pairs / edges:", n_noncommuting_pairs)
print("Number of commuting pairs:", n_commuting_pairs)
print("Is graph bipartite?", nx.is_bipartite(G))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G.degree[i],
            "commutes_with_all": G.degree[i] == 0,
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df = pd.DataFrame(vertex_rows)

print("\n=== Vertices: fermionic terms ===")
display(vertex_df)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": f"[T_{i}, T_{j}] != 0",
            "commutator": data["commutator_string"],
        }
    )

edge_df = pd.DataFrame(edge_rows)

print("\n=== Edges: noncommuting pairs ===")
display(edge_df)


# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G)

# node_labels = {
#     i: f"T_{i}"
#     for i in G.nodes()
# }

# node_sizes = [
#     1000 + 250 * G.degree[i]
#     for i in G.nodes()
# ]

# nx.draw_networkx_nodes(G, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G, pos, width=1.5)
# nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=11, font_weight="bold")

# plt.title("H2 Fermionic Noncommutation Graph")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph summary ===
Number of vertices / fermionic terms: 386
Number of total unordered pairs: 74305
Number of noncommuting pairs / edges: 35170
Number of commuting pairs: 39135
Is graph bipartite? False

=== Vertices: fermionic terms ===


,vertex,degree,commutes_with_all,fermionic_term
0,T_0,0,True,+3.39216161 I
1,T_1,84,False,-8.65361406 a_0^dagger a_0
2,T_2,166,False,+0.22578817 a_0^dagger a_2 + +0.22578817 a_2^dagger a_0
3,T_3,166,False,+0.19337590 a_0^dagger a_10 + +0.19337590 a_10^dagger a_0
4,T_4,84,False,-8.65361406 a_1^dagger a_1
...,...,...,...,...
381,T_381,242,False,+0.14081677 a_12^dagger a_11^dagger a_13 a_10 + +0.14081677 a_13^dagger a_10^dagger a_12 a_11
382,T_382,134,False,-0.43649838 a_13^dagger a_10^dagger a_13 a_10
383,T_383,134,False,-0.43649838 a_12^dagger a_11^dagger a_12 a_11
384,T_384,124,False,-0.29568160 a_13^dagger a_11^dagger a_13 a_11



=== Edges: noncommuting pairs ===


,source,target,meaning,commutator
0,T_1,T_2,"[T_1, T_2] != 0",-1.95388369 a_0^dagger a_2 + +1.95388369 a_2^dagger a_0
1,T_1,T_3,"[T_1, T_3] != 0",-1.67340044 a_0^dagger a_10 + +1.67340044 a_10^dagger a_0
2,T_1,T_24,"[T_1, T_24] != 0",+1.72316617 a_1^dagger a_0^dagger a_2 a_1 + -1.72316617 a_2^dagger a_1^dagger a_1 a_0
3,T_1,T_25,"[T_1, T_25] != 0",+1.56558003 a_1^dagger a_0^dagger a_10 a_1 + -1.56558003 a_10^dagger a_1^dagger a_1 a_0
4,T_1,T_27,"[T_1, T_27] != 0",+0.23180096 a_1^dagger a_0^dagger a_3 a_2 + -0.23180096 a_3^dagger a_2^dagger a_1 a_0
...,...,...,...,...
35165,T_377,T_381,"[T_377, T_381] != 0",-0.05278975 a_12^dagger a_11^dagger a_9^dagger a_13 a_10 a_9 + +0.05278975 a_13^dagger a_10^dagger a_9^dagger a_12 a_11 a_9
35166,T_378,T_379,"[T_378, T_379] != 0",-0.05803339 a_11^dagger a_10^dagger a_13 a_12 + +0.05803339 a_13^dagger a_12^dagger a_11 a_10
35167,T_379,T_385,"[T_379, T_385] != 0",-0.06895128 a_11^dagger a_10^dagger a_13 a_12 + +0.06895128 a_13^dagger a_12^dagger a_11 a_10
35168,T_381,T_382,"[T_381, T_382] != 0",+0.06146629 a_12^dagger a_11^dagger a_13 a_10 + -0.06146629 a_13^dagger a_10^dagger a_12 a_11


In [3]:
# Cell 2 alpha: Faster H2 / molecular fermionic noncommutation graph
#
# Main idea:
#   1. Use cheap fermionic index rules first.
#   2. Only if rules cannot decide, use exact OpenFermion symbolic commutator.
#   3. Never build sparse/dense matrices.
#
# This cell assumes Cell 1 already defined:
#   hermitian_terms
#   operator_to_string

import time
from collections import Counter

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from openfermion.utils import commutator
from openfermion.transforms import normal_ordered

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# Fermionic key utilities
# ------------------------------------------------------------

def key_modes(key):
    """
    Modes appearing in one OpenFermion monomial key.

    Example:
        ((3, 1), (0, 1), (3, 0), (0, 0)) -> {0, 3}
    """
    return frozenset(mode for mode, action in key)


def key_creations(key):
    """
    Creation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 1)


def key_annihilations(key):
    """
    Annihilation modes in one monomial.
    """
    return frozenset(mode for mode, action in key if action == 0)


def key_net_delta(key):
    """
    Net occupation change caused by a monomial.

    creation contributes +1
    annihilation contributes -1

    For example:
        a_2^dagger a_1^dagger a_3 a_0
    has delta:
        +1 on modes 2 and 1
        -1 on modes 3 and 0
    """
    delta = Counter()

    for mode, action in key:
        if action == 1:
            delta[mode] += 1
        else:
            delta[mode] -= 1

    return delta


def is_diagonal_key(key):
    """
    True if a monomial preserves occupation mode-by-mode.

    Examples:
        a_p^dagger a_p is diagonal.
        a_p^dagger a_q^dagger a_q a_p is diagonal.
        a_p^dagger a_q is not diagonal when p != q.
    """
    delta = key_net_delta(key)
    return all(value == 0 for value in delta.values())


def is_even_key(key):
    """
    Electronic Hamiltonian terms normally have even fermionic parity:
    length 0, 2, or 4.
    """
    return len(key) % 2 == 0


# ------------------------------------------------------------
# Safe monomial-level commutation rules
# ------------------------------------------------------------

def diagonal_key_commutes_with_key(diagonal_key, other_key):
    """
    Safe rule:

    A diagonal occupation operator depending on modes S commutes with another
    monomial if the other monomial has zero net occupation change on every
    mode in S.
    """
    support = key_modes(diagonal_key)
    delta = key_net_delta(other_key)

    return all(delta.get(mode, 0) == 0 for mode in support)


def no_cross_contractions_even_commute(key_a, key_b):
    """
    Safe rule for normal-ordered even fermionic monomials.

    If there are no possible cross contractions:
        annihilations(A) intersect creations(B) = empty
        annihilations(B) intersect creations(A) = empty

    then even monomials commute.

    This catches many cases beyond completely disjoint support.
    """
    if not is_even_key(key_a) or not is_even_key(key_b):
        return False

    a_ann = key_annihilations(key_a)
    a_cre = key_creations(key_a)

    b_ann = key_annihilations(key_b)
    b_cre = key_creations(key_b)

    return a_ann.isdisjoint(b_cre) and b_ann.isdisjoint(a_cre)


def monomial_pair_definitely_commutes(key_a, key_b):
    """
    Return (True, reason) only when we are sure two monomials commute.
    Return (False, None) if the rule cannot decide.

    Important:
        False here does NOT mean noncommuting.
        It only means "unknown; use exact symbolic fallback."
    """
    # Identity commutes with everything.
    if key_a == () or key_b == ():
        return True, "identity"

    # Any monomial commutes with itself.
    if key_a == key_b:
        return True, "same_monomial"

    # Diagonal occupation-like monomials commute with each other.
    if is_diagonal_key(key_a) and is_diagonal_key(key_b):
        return True, "diagonal_diagonal"

    # Diagonal with excitation-like term, if excitation preserves diagonal support.
    if is_diagonal_key(key_a) and diagonal_key_commutes_with_key(key_a, key_b):
        return True, "diagonal_support_preserved"

    if is_diagonal_key(key_b) and diagonal_key_commutes_with_key(key_b, key_a):
        return True, "diagonal_support_preserved"

    # Even monomials with no cross contractions commute.
    if no_cross_contractions_even_commute(key_a, key_b):
        return True, "no_cross_contractions_even"

    return False, None


# ------------------------------------------------------------
# Operator-level metadata and precheck
# ------------------------------------------------------------

def operator_metadata(op):
    """
    Precompute simple structural data for one FermionOperator.
    """
    keys = list(op.terms.keys())

    modes = set()
    for key in keys:
        modes.update(key_modes(key))

    return {
        "is_zero": len(keys) == 0,
        "only_identity": len(keys) == 1 and keys[0] == (),
        "modes": frozenset(modes),
        "is_even": all(is_even_key(key) for key in keys),
        "is_diagonal": all(is_diagonal_key(key) for key in keys),
        "number_of_monomials": len(keys),
    }


def operator_pair_definitely_commutes(A, B, meta_A, meta_B):
    """
    Return (True, reason) only for guaranteed-commuting pairs.
    Return (False, None) when unresolved.
    """
    if meta_A["is_zero"] or meta_B["is_zero"]:
        return True, "zero"

    if meta_A["only_identity"] or meta_B["only_identity"]:
        return True, "identity"

    # Very cheap global rule:
    # disjoint even fermionic operators commute.
    if (
        meta_A["is_even"]
        and meta_B["is_even"]
        and meta_A["modes"].isdisjoint(meta_B["modes"])
    ):
        return True, "disjoint_even_support"

    # Diagonal occupation-like operators commute with each other.
    if meta_A["is_diagonal"] and meta_B["is_diagonal"]:
        return True, "diagonal_diagonal"

    # More detailed but still cheap:
    # if every monomial pair has a safe commuting reason, the sums commute.
    reasons = Counter()

    for key_a in A.terms:
        for key_b in B.terms:
            ok, reason = monomial_pair_definitely_commutes(key_a, key_b)

            if not ok:
                return False, None

            reasons[reason] += 1

    if len(reasons) > 0:
        main_reason = reasons.most_common(1)[0][0]
        return True, f"all_monomial_pairs_{main_reason}"

    return False, None


# ------------------------------------------------------------
# Exact symbolic fallback
# ------------------------------------------------------------

def exact_symbolic_fermionic_commutator(A, B, tol=1e-12):
    """
    Exact symbolic commutator in fermionic algebra.

    This does not build a 2^n matrix.
    """
    C = normal_ordered(commutator(A, B))
    C.compress(abs_tol=tol)
    return C


# ------------------------------------------------------------
# Alpha graph builder
# ------------------------------------------------------------

def build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
):
    """
    Build noncommutation graph using:
        fast safe index rules first,
        exact symbolic OpenFermion fallback only when needed.

    Parameters
    ----------
    hermitian_terms:
        list of FermionOperator terms T_i from Cell 1.

    tol:
        numerical compression tolerance.

    store_commutators:
        False is recommended for large molecules.
        True is useful for H2 debugging, but can be memory-heavy.

    Returns
    -------
    G:
        networkx.Graph

    stats_df:
        pandas.DataFrame with timing and skip counts
    """
    t_start = time.perf_counter()

    G = nx.Graph()
    stats = Counter()

    metadata = [operator_metadata(T) for T in hermitian_terms]

    # Add vertices.
    for i, T in enumerate(hermitian_terms):
        G.add_node(
            i,
            label=f"T_{i}",
            operator=T,
            operator_string=operator_to_string(T),
            number_of_monomials=len(T.terms),
            modes=sorted(metadata[i]["modes"]),
            is_diagonal=metadata[i]["is_diagonal"],
            is_even=metadata[i]["is_even"],
        )

    n = len(hermitian_terms)

    # Pairwise graph construction.
    for i in range(n):
        A = hermitian_terms[i]
        meta_A = metadata[i]

        for j in range(i + 1, n):
            B = hermitian_terms[j]
            meta_B = metadata[j]

            stats["total_pairs"] += 1

            # 1. Fast guaranteed-commuting rules.
            definitely_commutes, reason = operator_pair_definitely_commutes(
                A, B, meta_A, meta_B
            )

            if definitely_commutes:
                stats["pairs_skipped_by_index_rules"] += 1
                stats[f"skip_{reason}"] += 1
                continue

            # 2. Exact symbolic fallback.
            stats["pairs_sent_to_exact_symbolic"] += 1

            Cij = exact_symbolic_fermionic_commutator(A, B, tol=tol)

            if len(Cij.terms) != 0:
                stats["noncommuting_edges"] += 1

                edge_data = {
                    "method": "exact_symbolic_fallback",
                    "meaning": f"[T_{i}, T_{j}] != 0",
                }

                if store_commutators:
                    edge_data["commutator"] = Cij
                    edge_data["commutator_string"] = operator_to_string(Cij)
                else:
                    edge_data["commutator_string"] = (
                        "(not stored; rerun with store_commutators=True)"
                    )

                G.add_edge(i, j, **edge_data)

            else:
                stats["exact_symbolic_found_commuting"] += 1

    elapsed = time.perf_counter() - t_start

    stats["vertices"] = n
    stats["edges"] = G.number_of_edges()
    stats["commuting_pairs"] = stats["total_pairs"] - G.number_of_edges()
    stats["elapsed_seconds"] = elapsed

    stats_df = pd.DataFrame([dict(stats)])

    return G, stats_df


# ------------------------------------------------------------
# Run alpha graph builder
# ------------------------------------------------------------

# For H2 debugging, you can set store_commutators=True.
# For larger molecules, keep this False.
G_alpha, stats_df = build_fermionic_noncommutation_graph_alpha(
    hermitian_terms,
    tol=1e-12,
    store_commutators=False,
)

print("=== Fermionic noncommutation graph alpha summary ===")
display(stats_df)

print("Number of vertices / fermionic terms:", G_alpha.number_of_nodes())
print("Number of noncommuting pairs / edges:", G_alpha.number_of_edges())
print("Is graph bipartite?", nx.is_bipartite(G_alpha))


# ------------------------------------------------------------
# Vertex table
# ------------------------------------------------------------

vertex_rows = []

for i, data in G_alpha.nodes(data=True):
    vertex_rows.append(
        {
            "vertex": f"T_{i}",
            "degree": G_alpha.degree[i],
            "commutes_with_all": G_alpha.degree[i] == 0,
            "number_of_monomials": data["number_of_monomials"],
            "modes": data["modes"],
            "is_diagonal": data["is_diagonal"],
            "fermionic_term": data["operator_string"],
        }
    )

vertex_df_alpha = pd.DataFrame(vertex_rows)

print("\n=== Alpha vertices: fermionic terms ===")
display(vertex_df_alpha)


# ------------------------------------------------------------
# Edge table
# ------------------------------------------------------------

edge_rows = []

for i, j, data in G_alpha.edges(data=True):
    edge_rows.append(
        {
            "source": f"T_{i}",
            "target": f"T_{j}",
            "meaning": data["meaning"],
            "method": data["method"],
            "commutator": data["commutator_string"],
        }
    )

edge_df_alpha = pd.DataFrame(edge_rows)

print("\n=== Alpha edges: noncommuting pairs ===")
display(edge_df_alpha)

# The commutation graph is too large for this.

# # ------------------------------------------------------------
# # Draw graph
# # ------------------------------------------------------------

# plt.figure(figsize=(12, 8))

# pos = nx.kamada_kawai_layout(G_alpha)

# node_labels = {
#     i: f"T_{i}"
#     for i in G_alpha.nodes()
# }

# node_sizes = [
#     1000 + 250 * G_alpha.degree[i]
#     for i in G_alpha.nodes()
# ]

# nx.draw_networkx_nodes(G_alpha, pos, node_size=node_sizes)
# nx.draw_networkx_edges(G_alpha, pos, width=1.5)
# nx.draw_networkx_labels(
#     G_alpha,
#     pos,
#     labels=node_labels,
#     font_size=11,
#     font_weight="bold",
# )

# plt.title("Fermionic Noncommutation Graph Alpha")
# plt.axis("off")
# plt.show()

=== Fermionic noncommutation graph alpha summary ===


,total_pairs,pairs_skipped_by_index_rules,skip_identity,pairs_sent_to_exact_symbolic,noncommuting_edges,skip_disjoint_even_support,skip_diagonal_diagonal,skip_all_monomial_pairs_diagonal_support_preserved,exact_symbolic_found_commuting,vertices,edges,commuting_pairs,elapsed_seconds
0,74305,34700,385,39605,35170,31889,1274,1152,4435,386,35170,39135,2.227835


Number of vertices / fermionic terms: 386
Number of noncommuting pairs / edges: 35170
Is graph bipartite? False

=== Alpha vertices: fermionic terms ===


,vertex,degree,commutes_with_all,number_of_monomials,modes,is_diagonal,fermionic_term
0,T_0,0,True,1,[],True,+3.39216161 I
1,T_1,84,False,1,[0],True,-8.65361406 a_0^dagger a_0
2,T_2,166,False,2,"[0, 2]",False,+0.22578817 a_0^dagger a_2 + +0.22578817 a_2^dagger a_0
3,T_3,166,False,2,"[0, 10]",False,+0.19337590 a_0^dagger a_10 + +0.19337590 a_10^dagger a_0
4,T_4,84,False,1,[1],True,-8.65361406 a_1^dagger a_1
...,...,...,...,...,...,...,...
381,T_381,242,False,2,"[10, 11, 12, 13]",False,+0.14081677 a_12^dagger a_11^dagger a_13 a_10 + +0.14081677 a_13^dagger a_10^dagger a_12 a_11
382,T_382,134,False,1,"[10, 13]",True,-0.43649838 a_13^dagger a_10^dagger a_13 a_10
383,T_383,134,False,1,"[11, 12]",True,-0.43649838 a_12^dagger a_11^dagger a_12 a_11
384,T_384,124,False,1,"[11, 13]",True,-0.29568160 a_13^dagger a_11^dagger a_13 a_11



=== Alpha edges: noncommuting pairs ===


,source,target,meaning,method,commutator
0,T_1,T_2,"[T_1, T_2] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
1,T_1,T_3,"[T_1, T_3] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
2,T_1,T_24,"[T_1, T_24] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
3,T_1,T_25,"[T_1, T_25] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
4,T_1,T_27,"[T_1, T_27] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
...,...,...,...,...,...
35165,T_377,T_381,"[T_377, T_381] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
35166,T_378,T_379,"[T_378, T_379] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
35167,T_379,T_385,"[T_379, T_385] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)
35168,T_381,T_382,"[T_381, T_382] != 0",exact_symbolic_fallback,(not stored; rerun with store_commutators=True)


In [4]:
print("Same edge set?")
print(set(G.edges()) == set(G_alpha.edges()))

Same edge set?
True


In [5]:
# Cell 3: Color graph, build commuting blocks, map JW/BK, verify commutation

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from itertools import combinations
from openfermion.ops import FermionOperator
from openfermion.transforms import normal_ordered, jordan_wigner, bravyi_kitaev

pd.set_option("display.max_colwidth", None)


# ------------------------------------------------------------
# 1. Color the noncommutation graph
# ------------------------------------------------------------
# Since edges mean noncommutation, each color class is a commuting group.

coloring = nx.coloring.greedy_color(G, strategy="largest_first")

color_groups = {}

for node, color in coloring.items():
    color_groups.setdefault(color, []).append(node)

for color in color_groups:
    color_groups[color] = sorted(color_groups[color])

color_names = [
    "red",
    "blue",
    "green",
    "orange",
    "purple",
    "brown",
    "pink",
    "gray",
]

color_name = {
    color: color_names[color] if color < len(color_names) else f"color_{color}"
    for color in color_groups
}

num_grouped_terms = sum(len(nodes) for nodes in color_groups.values())

print("Number of fermionic terms / vertices:", G.number_of_nodes())
print("Number of colors / commuting groups:", len(color_groups))
print("Number of grouped terms:", num_grouped_terms)

assert num_grouped_terms == G.number_of_nodes()


# ------------------------------------------------------------
# 2. Verify each color group is mutually commuting
# ------------------------------------------------------------

def verify_commuting_group(nodes, tol=1e-12):
    for i, j in combinations(nodes, 2):
        A = G.nodes[i]["operator"]
        B = G.nodes[j]["operator"]

        if not commute(A, B, tol=tol):
            return False

    return True


group_summary_rows = []

for color, nodes in sorted(color_groups.items()):
    group_summary_rows.append(
        {
            "color_id": color,
            "color_name": color_name[color],
            "number_of_terms": len(nodes),
            "vertices": [f"T_{i}" for i in nodes],
            "verified_mutually_commuting": verify_commuting_group(nodes),
        }
    )

group_summary_df = pd.DataFrame(group_summary_rows)

print("\n=== Commuting groups from graph coloring ===")
display(group_summary_df)


# ------------------------------------------------------------
# 3. Build Hamiltonian pieces by color
# ------------------------------------------------------------

H_by_color = {}

for color, nodes in sorted(color_groups.items()):
    H_color = FermionOperator.zero()

    for node in nodes:
        H_color += G.nodes[node]["operator"]

    H_color = normal_ordered(H_color)
    H_color.compress(abs_tol=1e-12)

    H_by_color[color] = H_color


color_block_rows = []

for color, nodes in sorted(color_groups.items()):
    for local_index, node in enumerate(nodes, start=1):
        color_block_rows.append(
            {
                "color_block": f"H_{color_name[color]}",
                "local_term_name": f"{color_name[color][0].upper()}_{local_index}",
                "vertex": f"T_{node}",
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

color_block_df = pd.DataFrame(color_block_rows)

print("\n=== Which T_i belongs to which color block ===")
display(color_block_df)


print("\n=== Hamiltonian split by commuting color groups ===")

for color, H_color in H_by_color.items():
    nodes = color_groups[color]

    print("\n" + "=" * 80)
    print(f"H_{color_name[color]} consists of:")
    print(" + ".join([f"T_{node}" for node in nodes]))

    print(f"\nSummed operator H_{color_name[color]} =")
    print(H_color)


# ------------------------------------------------------------
# 4. Trotter ordering induced by color groups
# ------------------------------------------------------------

trotter_order = []

for color, nodes in sorted(color_groups.items()):
    for node in nodes:
        trotter_order.append(node)

print("\n=== Trotter order by commuting color groups ===")
print([f"T_{i}" for i in trotter_order])


# ------------------------------------------------------------
# 5. Helper functions for JW/BK output
# ------------------------------------------------------------

def sort_qubit_key(term):
    return (len(term), term)


def format_qubit_term(term):
    if term == ():
        return "I"

    return " ".join([f"{pauli}{qubit}" for qubit, pauli in term])


def qubit_coeff_to_str(c, digits=8):
    c = complex(c)
    if abs(c.imag) < 1e-12:
        return f"{c.real:+.{digits}f}"
    return f"{c.real:+.{digits}f}{c.imag:+.{digits}f}j"


def qubit_operator_to_string(op, digits=8):
    pieces = []

    for term, coeff in sorted(op.terms.items(), key=lambda item: sort_qubit_key(item[0])):
        pieces.append(f"{qubit_coeff_to_str(coeff, digits)} {format_qubit_term(term)}")

    if len(pieces) == 0:
        return "0"

    return " + ".join(pieces)


def apply_bk(op, n_qubits):
    try:
        return bravyi_kitaev(op, n_qubits=n_qubits)
    except TypeError:
        return bravyi_kitaev(op, n_qubits)


def qubit_commutator(A, B, tol=1e-12):
    C = A * B - B * A
    C.compress(abs_tol=tol)
    return C


def qubit_commute(A, B, tol=1e-12):
    C = qubit_commutator(A, B, tol=tol)
    return len(C.terms) == 0


n_qubits = molecule.n_qubits

print("\nNumber of qubits / spin orbitals:", n_qubits)


# ------------------------------------------------------------
# 6. Map each fermionic vertex T_i to JW(T_i) and BK(T_i)
# ------------------------------------------------------------

mapped_rows = []

for node in sorted(G.nodes()):
    T_i = G.nodes[node]["operator"]

    JW_T_i = jordan_wigner(T_i)
    JW_T_i.compress(abs_tol=1e-12)

    BK_T_i = apply_bk(T_i, n_qubits=n_qubits)
    BK_T_i.compress(abs_tol=1e-12)

    G.nodes[node]["JW_operator"] = JW_T_i
    G.nodes[node]["BK_operator"] = BK_T_i
    G.nodes[node]["JW_operator_string"] = qubit_operator_to_string(JW_T_i)
    G.nodes[node]["BK_operator_string"] = qubit_operator_to_string(BK_T_i)

    node_color = coloring[node]
    node_color_name = color_name[node_color]

    mapped_rows.append(
        {
            "color": node_color_name,
            "vertex": f"T_{node}",
            "fermionic_term": G.nodes[node]["operator_string"],
            "number_of_JW_Pauli_strings": len(JW_T_i.terms),
            "JW_transform": qubit_operator_to_string(JW_T_i),
            "number_of_BK_Pauli_strings": len(BK_T_i.terms),
            "BK_transform": qubit_operator_to_string(BK_T_i),
        }
    )

mapped_terms_df = pd.DataFrame(mapped_rows)

print("\n=== Fermionic terms mapped to JW and BK ===")
display(mapped_terms_df)


# ------------------------------------------------------------
# 7. Map each color block H_color to JW and BK
# ------------------------------------------------------------

JW_by_color = {}
BK_by_color = {}

block_rows = []

for color, H_color in sorted(H_by_color.items()):
    JW_color = jordan_wigner(H_color)
    JW_color.compress(abs_tol=1e-12)

    BK_color = apply_bk(H_color, n_qubits=n_qubits)
    BK_color.compress(abs_tol=1e-12)

    JW_by_color[color] = JW_color
    BK_by_color[color] = BK_color

    block_rows.append(
        {
            "color_block": f"H_{color_name[color]}",
            "fermionic_vertices": " + ".join([f"T_{node}" for node in color_groups[color]]),
            "number_of_fermionic_terms": len(color_groups[color]),
            "number_of_JW_Pauli_strings": len(JW_color.terms),
            "JW_block": qubit_operator_to_string(JW_color),
            "number_of_BK_Pauli_strings": len(BK_color.terms),
            "BK_block": qubit_operator_to_string(BK_color),
        }
    )

block_map_df = pd.DataFrame(block_rows)

print("\n=== Color blocks mapped to JW and BK ===")
display(block_map_df)


# ------------------------------------------------------------
# 8. Verify same-color terms commute after JW and BK
# ------------------------------------------------------------

verification_rows = []

for color, nodes in sorted(color_groups.items()):
    for i, j in combinations(nodes, 2):
        Ti = G.nodes[i]["operator"]
        Tj = G.nodes[j]["operator"]

        JW_Ti = G.nodes[i]["JW_operator"]
        JW_Tj = G.nodes[j]["JW_operator"]

        BK_Ti = G.nodes[i]["BK_operator"]
        BK_Tj = G.nodes[j]["BK_operator"]

        verification_rows.append(
            {
                "color_group": color_name[color],
                "pair": f"T_{i}, T_{j}",
                "fermionic_commute": commute(Ti, Tj),
                "JW_commute": qubit_commute(JW_Ti, JW_Tj),
                "BK_commute": qubit_commute(BK_Ti, BK_Tj),
            }
        )

verification_df = pd.DataFrame(verification_rows)

print("\n=== Verify commuting groups after JW/BK mapping ===")
display(verification_df)

Number of fermionic terms / vertices: 386
Number of colors / commuting groups: 45
Number of grouped terms: 386

=== Commuting groups from graph coloring ===


,color_id,color_name,number_of_terms,vertices,verified_mutually_commuting
0,0,red,14,"[T_0, T_17, T_18, T_29, T_59, T_62, T_153, T_209, T_249, T_348, T_353, T_354, T_362, T_365]",True
1,1,blue,14,"[T_15, T_16, T_30, T_63, T_109, T_150, T_208, T_248, T_290, T_346, T_366, T_368, T_369, T_375]",True
2,2,green,14,"[T_19, T_20, T_36, T_56, T_113, T_196, T_203, T_215, T_245, T_347, T_350, T_351, T_360, T_378]",True
3,3,orange,14,"[T_37, T_108, T_114, T_195, T_202, T_210, T_214, T_261, T_352, T_355, T_359, T_361, T_370, T_374]",True
4,4,purple,4,"[T_47, T_64, T_135, T_187]",True
5,5,brown,4,"[T_48, T_65, T_128, T_141]",True
6,6,pink,4,"[T_54, T_115, T_134, T_186]",True
7,7,gray,4,"[T_55, T_116, T_127, T_140]",True
8,8,color_8,10,"[T_61, T_69, T_99, T_182, T_191, T_198, T_204, T_222, T_224, T_268]",True
9,9,color_9,10,"[T_70, T_100, T_111, T_146, T_175, T_205, T_228, T_230, T_273, T_338]",True



=== Which T_i belongs to which color block ===


,color_block,local_term_name,vertex,fermionic_term
0,H_red,R_1,T_0,+3.39216161 I
1,H_red,R_2,T_17,-2.29974267 a_8^dagger a_8
2,H_red,R_3,T_18,-2.29974267 a_9^dagger a_9
3,H_red,R_4,T_29,-0.00604787 a_1^dagger a_0^dagger a_5 a_4 + -0.00604787 a_5^dagger a_4^dagger a_1 a_0
4,H_red,R_5,T_59,+0.02566293 a_4^dagger a_0^dagger a_12 a_2 + +0.02566293 a_12^dagger a_2^dagger a_4 a_0
...,...,...,...,...
381,H_color_42,C_2,T_278,+0.04743776 a_9^dagger a_8^dagger a_10 a_3 + +0.04743776 a_10^dagger a_3^dagger a_9 a_8
382,H_color_43,C_1,T_332,-0.01377638 a_8^dagger a_5^dagger a_12 a_9 + -0.01377638 a_12^dagger a_9^dagger a_8 a_5
383,H_color_43,C_2,T_341,+0.01377638 a_9^dagger a_8^dagger a_12 a_5 + +0.01377638 a_12^dagger a_5^dagger a_9 a_8
384,H_color_44,C_1,T_367,-0.01653154 a_9^dagger a_8^dagger a_13 a_12 + -0.01653154 a_13^dagger a_12^dagger a_9 a_8



=== Hamiltonian split by commuting color groups ===

H_red consists of:
T_0 + T_17 + T_18 + T_29 + T_59 + T_62 + T_153 + T_209 + T_249 + T_348 + T_353 + T_354 + T_362 + T_365

Summed operator H_red =
3.3921616084615387 [] +
-0.006047866793733274 [1^ 0^ 5 4] +
-0.056584600456812524 [3^ 2^ 13 12] +
0.025662934515149095 [4^ 0^ 12 2] +
0.006047866793733274 [4^ 1^ 5 0] +
0.006047866793733274 [5^ 0^ 4 1] +
0.025662934515149095 [5^ 1^ 13 3] +
-0.006047866793733274 [5^ 4^ 1 0] +
-0.05092314430548645 [7^ 6^ 11 10] +
-2.2997426657607067 [8^ 8] +
-0.4498590410866713 [9^ 8^ 9 8] +
-2.2997426657607067 [9^ 9] +
-0.3167299451888317 [10^ 6^ 10 6] +
0.05092314430548645 [10^ 7^ 11 6] +
0.05092314430548645 [11^ 6^ 10 7] +
-0.3167299451888317 [11^ 7^ 11 7] +
-0.05092314430548645 [11^ 10^ 7 6] +
0.025662934515149095 [12^ 2^ 4 0] +
0.056584600456812524 [12^ 3^ 13 2] +
0.056584600456812524 [13^ 2^ 12 3] +
0.025662934515149095 [13^ 3^ 5 1] +
-0.056584600456812524 [13^ 12^ 3 2]

H_blue consists of:
T_15 + T_1

,color,vertex,fermionic_term,number_of_JW_Pauli_strings,JW_transform,number_of_BK_Pauli_strings,BK_transform
0,red,T_0,+3.39216161 I,1,+3.39216161 I,1,+3.39216161 I
1,color_14,T_1,-8.65361406 a_0^dagger a_0,2,-4.32680703 I + +4.32680703 Z0,2,-4.32680703 I + +4.32680703 Z0
2,color_17,T_2,+0.22578817 a_0^dagger a_2 + +0.22578817 a_2^dagger a_0,2,+0.11289409 X0 Z1 X2 + +0.11289409 Y0 Z1 Y2,2,+0.11289409 X0 Y1 Y2 + -0.11289409 Y0 Y1 X2
3,color_31,T_3,+0.19337590 a_0^dagger a_10 + +0.19337590 a_10^dagger a_0,2,+0.09668795 X0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 X10 + +0.09668795 Y0 Z1 Z2 Z3 Z4 Z5 Z6 Z7 Z8 Z9 Y10,2,+0.09668795 X0 X1 X3 Y7 Z9 Y10 X11 + -0.09668795 Y0 X1 X3 Y7 Z9 X10 X11
4,color_14,T_4,-8.65361406 a_1^dagger a_1,2,-4.32680703 I + +4.32680703 Z1,2,-4.32680703 I + +4.32680703 Z0 Z1
...,...,...,...,...,...,...,...
381,color_19,T_381,+0.14081677 a_12^dagger a_11^dagger a_13 a_10 + +0.14081677 a_13^dagger a_10^dagger a_12 a_11,8,-0.01760210 X10 X11 X12 X13 + -0.01760210 X10 X11 Y12 Y13 + -0.01760210 X10 Y11 X12 Y13 + +0.01760210 X10 Y11 Y12 X13 + +0.01760210 Y10 X11 X12 Y13 + -0.01760210 Y10 X11 Y12 X13 + -0.01760210 Y10 Y11 X12 X13 + -0.01760210 Y10 Y11 Y12 Y13,8,-0.01760210 X10 X12 + -0.01760210 Y10 Y12 + +0.01760210 X10 X12 Z13 + +0.01760210 Y10 Y12 Z13 + +0.01760210 Z9 X10 Z11 X12 + +0.01760210 Z9 Y10 Z11 Y12 + -0.01760210 Z9 X10 Z11 X12 Z13 + -0.01760210 Z9 Y10 Z11 Y12 Z13
382,color_32,T_382,-0.43649838 a_13^dagger a_10^dagger a_13 a_10,4,+0.10912459 I + -0.10912459 Z10 + -0.10912459 Z13 + +0.10912459 Z10 Z13,4,+0.10912459 I + -0.10912459 Z10 + -0.10912459 Z12 Z13 + +0.10912459 Z10 Z12 Z13
383,color_31,T_383,-0.43649838 a_12^dagger a_11^dagger a_12 a_11,4,+0.10912459 I + -0.10912459 Z11 + -0.10912459 Z12 + +0.10912459 Z11 Z12,4,+0.10912459 I + -0.10912459 Z12 + -0.10912459 Z9 Z10 Z11 + +0.10912459 Z9 Z10 Z11 Z12
384,color_11,T_384,-0.29568160 a_13^dagger a_11^dagger a_13 a_11,4,+0.07392040 I + -0.07392040 Z11 + -0.07392040 Z13 + +0.07392040 Z11 Z13,4,+0.07392040 I + -0.07392040 Z12 Z13 + -0.07392040 Z9 Z10 Z11 + +0.07392040 Z9 Z10 Z11 Z12 Z13



=== Color blocks mapped to JW and BK ===


,color_block,fermionic_vertices,number_of_fermionic_terms,number_of_JW_Pauli_strings,JW_block,number_of_BK_Pauli_strings,BK_block
0,H_red,T_0 + T_17 + T_18 + T_29 + T_59 + T_62 + T_153 + T_209 + T_249 + T_348 + T_353 + T_354 + T_362 + T_365,14,38,+1.36324868 I + -0.07918249 Z6 + -0.07918249 Z7 + +1.03740657 Z8 + +1.03740657 Z9 + -0.07918249 Z10 + -0.07918249 Z11 + +0.07918249 Z6 Z10 + +0.07918249 Z7 Z11 + +0.11246476 Z8 Z9 + -0.00151197 X0 X1 Y4 Y5 + +0.00151197 X0 Y1 Y4 X5 + +0.00151197 Y0 X1 X4 Y5 + -0.00151197 Y0 Y1 X4 X5 + -0.01414615 X2 X3 Y12 Y13 + +0.01414615 X2 Y3 Y12 X13 + +0.01414615 Y2 X3 X12 Y13 + -0.01414615 Y2 Y3 X12 X13 + -0.01273079 X6 X7 Y10 Y11 + +0.01273079 X6 Y7 Y10 X11 + +0.01273079 Y6 X7 X10 Y11 + -0.01273079 Y6 Y7 X10 X11 + -0.00320787 X0 Z1 X2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00320787 X0 Z1 X2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + +0.00320787 X0 Z1 Y2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00320787 X0 Z1 Y2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00320787 Y0 Z1 X2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + +0.00320787 Y0 Z1 X2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00320787 Y0 Z1 Y2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00320787 Y0 Z1 Y2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00320787 X1 Z2 X3 X5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 X13 + -0.00320787 X1 Z2 X3 Y5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13 + +0.00320787 X1 Z2 Y3 X5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13 + -0.00320787 X1 Z2 Y3 Y5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 X13 + -0.00320787 Y1 Z2 X3 X5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13 + +0.00320787 Y1 Z2 X3 Y5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 X13 + -0.00320787 Y1 Z2 Y3 X5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 X13 + -0.00320787 Y1 Z2 Y3 Y5 Z6 Z7 Z8 Z9 Z10 Z11 Z12 Y13,38,+1.36324868 I + -0.07918249 Z6 + +1.03740657 Z8 + +0.11246476 Z9 + -0.07918249 Z10 + +0.07918249 Z6 Z10 + +1.03740657 Z8 Z9 + +0.00151197 X0 Z1 X4 + +0.00151197 X0 X4 Z5 + +0.00151197 Y0 Z1 Y4 + +0.00151197 Y0 Y4 Z5 + +0.01414615 X2 X12 Z13 + +0.01414615 Y2 Y12 Z13 + -0.07918249 Z9 Z10 Z11 + +0.01414615 Z1 X2 Z3 X12 + +0.01414615 Z1 Y2 Z3 Y12 + -0.07918249 Z3 Z5 Z6 Z7 + +0.01273079 X6 Z9 X10 Z11 + +0.01273079 Y6 Z9 Y10 Z11 + -0.00320787 Y1 Y5 Y7 Z11 Y13 + +0.01273079 Z3 Z5 X6 Z7 X10 + +0.01273079 Z3 Z5 Y6 Z7 Y10 + +0.00320787 Z0 X1 Z4 X5 Y7 Z11 Y13 + -0.00320787 Z0 X1 Y5 Y7 Z11 Z12 X13 + -0.00320787 Y1 Z4 X5 Y7 Z11 Z12 X13 + +0.07918249 Z3 Z5 Z6 Z7 Z9 Z10 Z11 + +0.00320787 Z0 Y1 Z2 Z3 Y5 Y7 Z11 Y13 + -0.00320787 X1 Z2 Z3 Z4 X5 Y7 Z11 Y13 + +0.00320787 X1 Z2 Z3 Y5 Y7 Z11 Z12 X13 + +0.00320787 X0 Y1 X2 Z3 X4 X5 Y7 Z11 X12 X13 + +0.00320787 X0 Y1 X2 Z3 Y4 X5 Y7 Z11 Y12 X13 + -0.00320787 X0 Y1 Y2 Z3 X4 X5 Y7 Z11 Y12 X13 + +0.00320787 X0 Y1 Y2 Z3 Y4 X5 Y7 Z11 X12 X13 + +0.00320787 Y0 Y1 X2 Z3 X4 X5 Y7 Z11 Y12 X13 + -0.00320787 Y0 Y1 X2 Z3 Y4 X5 Y7 Z11 X12 X13 + +0.00320787 Y0 Y1 Y2 Z3 X4 X5 Y7 Z11 X12 X13 + +0.00320787 Y0 Y1 Y2 Z3 Y4 X5 Y7 Z11 Y12 X13 + +0.00320787 Z0 Y1 Z2 Z3 Z4 X5 Y7 Z11 Z12 X13
1,H_blue,T_15 + T_16 + T_30 + T_63 + T_109 + T_150 + T_208 + T_248 + T_290 + T_346 + T_366 + T_368 + T_369 + T_375,14,36,-1.82258505 I + -0.11327601 Z1 + -0.09305187 Z3 + -0.11327601 Z5 + +1.03740657 Z6 + +1.03740657 Z7 + -0.07918249 Z8 + -0.07918249 Z9 + -0.07918249 Z10 + -0.07918249 Z11 + -0.09305187 Z13 + +0.11327601 Z1 Z5 + +0.09305187 Z3 Z13 + +0.11246476 Z6 Z7 + +0.07918249 Z8 Z10 + +0.07918249 Z9 Z11 + -0.01273079 X8 X9 Y10 Y11 + +0.01273079 X8 Y9 Y10 X11 + +0.01273079 Y8 X9 X10 Y11 + -0.01273079 Y8 Y9 X10 X11 + -0.00281703 X0 X1 X5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00281703 X0 Y1 Y5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00281703 Y0 X1 X5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00281703 Y0 Y1 Y5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00107443 X0 Z1 X2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00107443 X0 Z1 X2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00107443 X0 Z1 Y2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + +0.00107443 X0 Z1 Y2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + +0.00107443 Y0 Z1 X2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + -0.00107443 Y0 Z1 X2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00107443 Y0 Z1 Y2 X4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 X12 + -0.00107443 Y0 Z1 Y2 Y4 Z5 Z6 Z7 Z8 Z9 Z10 Z11 Y12 + +0.01110473 X2 X3 Y4 Z5 Z6 Z7 Z8 Z9 Z


=== Verify commuting groups after JW/BK mapping ===


,color_group,pair,fermionic_commute,JW_commute,BK_commute
0,red,"T_0, T_17",True,True,True
1,red,"T_0, T_18",True,True,True
2,red,"T_0, T_29",True,True,True
3,red,"T_0, T_59",True,True,True
4,red,"T_0, T_62",True,True,True
...,...,...,...,...,...
2229,color_40,"T_168, T_177",True,True,True
2230,color_41,"T_233, T_240",True,True,True
2231,color_42,"T_272, T_278",True,True,True
2232,color_43,"T_332, T_341",True,True,True


In [6]:
# Find duplicated JW Pauli strings across fermionic terms T_i

from collections import defaultdict
import pandas as pd

pauli_usage = defaultdict(list)

for node in sorted(G.nodes()):
    JW_T = G.nodes[node]["JW_operator"]

    for pauli_key, coeff in JW_T.terms.items():
        pauli_string = format_qubit_term(pauli_key)

        pauli_usage[pauli_string].append(
            {
                "vertex": f"T_{node}",
                "coefficient": coeff,
                "fermionic_term": G.nodes[node]["operator_string"],
            }
        )

duplicate_rows = []

for pauli_string, appearances in pauli_usage.items():
    if len(appearances) > 1:
        duplicate_rows.append(
            {
                "JW_Pauli_string": pauli_string,
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "coefficients": [x["coefficient"] for x in appearances],
            }
        )

duplicate_jw_df = pd.DataFrame(duplicate_rows)
duplicate_jw_df = duplicate_jw_df.sort_values(
    "number_of_appearances",
    ascending=False
).reset_index(drop=True)

print("Total JW Pauli-string appearances:", sum(len(G.nodes[node]["JW_operator"].terms) for node in G.nodes()))
print("Number of unique JW Pauli strings:", len(pauli_usage))
print("Number of duplicated JW Pauli strings:", len(duplicate_jw_df))

display(duplicate_jw_df)

Total JW Pauli-string appearances: 2201
Number of unique JW Pauli strings: 994
Number of duplicated JW Pauli strings: 711


,JW_Pauli_string,number_of_appearances,appears_in_vertices,coefficients
0,I,106,"[T_0, T_1, T_4, T_7, T_9, T_11, T_13, T_15, T_16, T_17, T_18, T_19, T_20, T_21, T_22, T_23, T_38, T_44, T_56, T_66, T_74, T_79, T_83, T_88, T_92, T_103, T_108, T_121, T_124, T_136, T_142, T_150, T_157, T_161, T_165, T_169, T_178, T_183, T_192, T_195, T_200, T_210, T_216, T_222, T_225, T_228, T_231, T_234, T_241, T_245, T_253, T_255, T_261, T_265, T_268, T_270, T_273, T_279, T_283, T_288, T_290, T_293, T_300, T_303, T_306, T_309, T_312, T_315, T_318, T_323, T_325, T_328, T_330, T_333, T_335, T_338, T_343, T_345, T_346, T_350, T_352, T_353, T_355, T_356, T_358, T_359, T_360, T_361, T_362, T_363, T_364, T_365, T_368, T_370, T_371, T_373, T_374, T_375, T_376, T_377, ...]","[3.3921616084615387, -4.326807030263166, -4.326807030263166, -1.2340546110559247, -1.2340546110559247, -1.215245368365108, -1.215245368365108, -1.1498713328803523, -1.1498713328803523, -1.1498713328803534, -1.1498713328803534, -0.9578827479006828, -0.9578827479006828, -0.8989090819408376, -0.8989090819408376, 0.5678722535171743, 0.11545479210392827, 0.1221514437504704, 0.11327600938641182, 0.11478797608484514, 0.1383515548866924, 0.14229336687143013, 0.13835155488669257, 0.1422933668714303, 0.11317552161416361, 0.1190723271583569, 0.13917930357851754, 0.14453738718522274, 0.1221514437504704, 0.11545479210392827, 0.11478797608484514, 0.11327600938641182, 0.14229336687143013, 0.1383515548866924, 0.1422933668714303, 0.13835155488669257, 0.1190723271583569, 0.11317552161416361, 0.14453738718522274, 0.13917930357851754, 0.0997579308471877, 0.061966708537915284, 0.10309654502523614, 0.08001574498509717, 0.09238728853855022, 0.08001574498509728, 0.09238728853855033, 0.07994055092939271, 0.09934427228949878, 0.09305187375107359, 0.10719802386527671, 0.10309654502523614, 0.061966708537915284, 0.09238728853855022, 0.08001574498509717, 0.09238728853855033, 0.08001574498509728, 0.09934427228949878, 0.07994055092939271, 0.10719802386527671, 0.09305187375107359, 0.10888573943004314, 0.08557179384084701, 0.08924796546039956, 0.08557179384084711, 0.08924796546039968, 0.08124465742865461, 0.1021021909745089, 0.09131649839841018, 0.11190044845328519, 0.08924796546039956, 0.08557179384084701, 0.08924796546039968, 0.08557179384084711, 0.1021021909745089, 0.08124465742865461, 0.11190044845328519, 0.09131649839841018, 0.11246476027166757, 0.09427772585578936, 0.10034007066108216, 0.07918248629720792, 0.09191327237357955, 0.09372064187159158, 0.09785352802640564, 0.10034007066108216, 0.09427772585578936, 0.09191327237357955, 0.07918248629720792, 0.09785352802640564, 0.09372064187159158, 0.11246476027166782, 0.079182486297208, 0.09191327237357963, 0.09372064187159165, 0.0978535280264057, 0.09191327237357963, 0.079182486297208, 0.0978535280264057, 0.09372064187159165, ...]"
1,Z9,14,"[T_18, T_88, T_169, T_231, T_273, T_309, T_333, T_352, T_360, T_365, T_374, T_375, T_376, T_377]","[1.1498713328803534, -0.1422933668714303, -0.13835155488669257, -0.09238728853855033, -0.08001574498509728, -0.08924796546039968, -0.08557179384084711, -0.10034007066108216, -0.09427772585578936, -0.11246476027166782, -0.09191327237357963, -0.079182486297208, -0.0978535280264057, -0.09372064187159165]"
2,Z5,14,"[T_13, T_66, T_150, T_216, T_261, T_293, T_325, T_328, T_330, T_333, T_335, T_338, T_343, T_345]","[1.215245368365108, -0.11478797608484514, -0.11327600938641182, -0.10309654502523614, -0.061966708537915284, -0.10888573943004314, -0.08924796546039956, -0.08557179384084701, -0.08924796546039968, -0.08557179384084711, -0.1021021909745089, -0.08124465742865461, -0.11190044845328519, -0.09131649839841018]"
3,Z3,14,"[T_9, T_44, T_136, T_200, T_255, T_261, T_265, T_268, T_270, T_273, T_279, T_283, T_288, T_290]","[1.2340546110559247, -0.1221514437504704, -0.11545479210392827, -0.0997579308471877, -0.10309654502523614, -0.061966708537915284, -0.09238728853855022, -0.08001574498509717, -0.09238728853855033, -0.08001574498509728, -0.

In [7]:
# Compute Pauli-duplication ratio for JW and BK

from collections import defaultdict
import pandas as pd

from openfermion.ops import FermionOperator
from openfermion.transforms import jordan_wigner, bravyi_kitaev, normal_ordered


def pauli_support(qubit_op, tol=1e-12, include_identity=True):
    """
    Return the set of Pauli strings with nonzero coefficients.
    """
    qubit_op.compress(abs_tol=tol)

    support = set()

    for pauli_key, coeff in qubit_op.terms.items():
        if abs(coeff) <= tol:
            continue

        if not include_identity and pauli_key == ():
            continue

        support.add(pauli_key)

    return support


def map_fermion_to_qubit(op, mapping="JW", n_qubits=None):
    """
    Map a FermionOperator to a QubitOperator using JW or BK.
    """
    mapping = mapping.upper()

    if mapping == "JW":
        qop = jordan_wigner(op)

    elif mapping == "BK":
        if n_qubits is None:
            raise ValueError("n_qubits is required for BK.")

        try:
            qop = bravyi_kitaev(op, n_qubits=n_qubits)
        except TypeError:
            qop = bravyi_kitaev(op, n_qubits)

    else:
        raise ValueError("mapping must be 'JW' or 'BK'.")

    qop.compress(abs_tol=1e-12)
    return qop


def pauli_duplication_ratio(
    fermionic_terms,
    mapping="JW",
    n_qubits=None,
    include_identity=True,
    tol=1e-12,
):
    """
    Compute

        sum_alpha #mapping(H_alpha) / #mapping(H)

    where H = sum_alpha H_alpha.

    Also returns a duplicate-use table.
    """

    H_full = FermionOperator.zero()

    numerator = 0
    union_support = set()
    pauli_usage = defaultdict(list)

    for alpha, H_alpha in enumerate(fermionic_terms):
        H_full += H_alpha

        Q_alpha = map_fermion_to_qubit(
            H_alpha,
            mapping=mapping,
            n_qubits=n_qubits,
        )

        support_alpha = pauli_support(
            Q_alpha,
            tol=tol,
            include_identity=include_identity,
        )

        numerator += len(support_alpha)
        union_support |= support_alpha

        for pauli_key, coeff in Q_alpha.terms.items():
            if abs(coeff) <= tol:
                continue

            if not include_identity and pauli_key == ():
                continue

            pauli_usage[pauli_key].append(
                {
                    "vertex": f"T_{alpha}",
                    "coefficient": coeff,
                }
            )

    H_full = normal_ordered(H_full)
    H_full.compress(abs_tol=tol)

    Q_full = map_fermion_to_qubit(
        H_full,
        mapping=mapping,
        n_qubits=n_qubits,
    )

    full_support = pauli_support(
        Q_full,
        tol=tol,
        include_identity=include_identity,
    )

    denominator = len(full_support)

    duplication_ratio = numerator / denominator
    union_ratio = numerator / len(union_support)

    summary_df = pd.DataFrame(
        [
            {
                "mapping": mapping.upper(),
                "include_identity": include_identity,
                "sum_alpha_number_of_Pauli_strings": numerator,
                "number_of_unique_Pauli_strings_before_cancellation": len(union_support),
                "number_of_Pauli_strings_in_full_H": denominator,
                "duplication_ratio": duplication_ratio,
                "raw_reuse_ratio_before_cancellation": union_ratio,
            }
        ]
    )

    duplicate_rows = []

    for pauli_key, appearances in pauli_usage.items():
        if len(appearances) <= 1:
            continue

        combined_coeff = Q_full.terms.get(pauli_key, 0.0)

        duplicate_rows.append(
            {
                "Pauli_string": format_qubit_term(pauli_key),
                "number_of_appearances": len(appearances),
                "appears_in_vertices": [x["vertex"] for x in appearances],
                "individual_coefficients": [complex(x["coefficient"]) for x in appearances],
                "combined_coefficient_in_full_H": complex(combined_coeff),
                "survives_in_full_H": abs(combined_coeff) > tol,
            }
        )

    duplicate_df = pd.DataFrame(duplicate_rows)

    if len(duplicate_df) > 0:
        duplicate_df = duplicate_df.sort_values(
            "number_of_appearances",
            ascending=False,
        ).reset_index(drop=True)

    return summary_df, duplicate_df


# ------------------------------------------------------------
# Run for JW
# ------------------------------------------------------------

n_qubits = molecule.n_qubits

jw_summary_df, jw_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="JW",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== JW Pauli-duplication ratio ===")
display(jw_summary_df)

print("\n=== Duplicated JW Pauli strings ===")
display(jw_duplicate_df)


# ------------------------------------------------------------
# Optional: run for BK too
# ------------------------------------------------------------

bk_summary_df, bk_duplicate_df = pauli_duplication_ratio(
    hermitian_terms,
    mapping="BK",
    n_qubits=n_qubits,
    include_identity=True,
)

print("=== BK Pauli-duplication ratio ===")
display(bk_summary_df)

print("\n=== Duplicated BK Pauli strings ===")
display(bk_duplicate_df)

=== JW Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,JW,True,2201,994,666,3.304805,2.214286



=== Duplicated JW Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,106,"[T_0, T_1, T_4, T_7, T_9, T_11, T_13, T_15, T_16, T_17, T_18, T_19, T_20, T_21, T_22, T_23, T_38, T_44, T_56, T_66, T_74, T_79, T_83, T_88, T_92, T_103, T_108, T_121, T_124, T_136, T_142, T_150, T_157, T_161, T_165, T_169, T_178, T_183, T_192, T_195, T_200, T_210, T_216, T_222, T_225, T_228, T_231, T_234, T_241, T_245, T_253, T_255, T_261, T_265, T_268, T_270, T_273, T_279, T_283, T_288, T_290, T_293, T_300, T_303, T_306, T_309, T_312, T_315, T_318, T_323, T_325, T_328, T_330, T_333, T_335, T_338, T_343, T_345, T_346, T_350, T_352, T_353, T_355, T_356, T_358, T_359, T_360, T_361, T_362, T_363, T_364, T_365, T_368, T_370, T_371, T_373, T_374, T_375, T_376, T_377, ...]","[(3.3921616084615387+0j), (-4.326807030263166+0j), (-4.326807030263166+0j), (-1.2340546110559247+0j), (-1.2340546110559247+0j), (-1.215245368365108+0j), (-1.215245368365108+0j), (-1.1498713328803523+0j), (-1.1498713328803523+0j), (-1.1498713328803534+0j), (-1.1498713328803534+0j), (-0.9578827479006828+0j), (-0.9578827479006828+0j), (-0.8989090819408376+0j), (-0.8989090819408376+0j), (0.5678722535171743+0j), (0.11545479210392827+0j), (0.1221514437504704+0j), (0.11327600938641182+0j), (0.11478797608484514+0j), (0.1383515548866924+0j), (0.14229336687143013+0j), (0.13835155488669257+0j), (0.1422933668714303+0j), (0.11317552161416361+0j), (0.1190723271583569+0j), (0.13917930357851754+0j), (0.14453738718522274+0j), (0.1221514437504704+0j), (0.11545479210392827+0j), (0.11478797608484514+0j), (0.11327600938641182+0j), (0.14229336687143013+0j), (0.1383515548866924+0j), (0.1422933668714303+0j), (0.13835155488669257+0j), (0.1190723271583569+0j), (0.11317552161416361+0j), (0.14453738718522274+0j), (0.13917930357851754+0j), (0.0997579308471877+0j), (0.061966708537915284+0j), (0.10309654502523614+0j), (0.08001574498509717+0j), (0.09238728853855022+0j), (0.08001574498509728+0j), (0.09238728853855033+0j), (0.07994055092939271+0j), (0.09934427228949878+0j), (0.09305187375107359+0j), (0.10719802386527671+0j), (0.10309654502523614+0j), (0.061966708537915284+0j), (0.09238728853855022+0j), (0.08001574498509717+0j), (0.09238728853855033+0j), (0.08001574498509728+0j), (0.09934427228949878+0j), (0.07994055092939271+0j), (0.10719802386527671+0j), (0.09305187375107359+0j), (0.10888573943004314+0j), (0.08557179384084701+0j), (0.08924796546039956+0j), (0.08557179384084711+0j), (0.08924796546039968+0j), (0.08124465742865461+0j), (0.1021021909745089+0j), (0.09131649839841018+0j), (0.11190044845328519+0j), (0.08924796546039956+0j), (0.08557179384084701+0j), (0.08924796546039968+0j), (0.08557179384084711+0j), (0.1021021909745089+0j), (0.08124465742865461+0j), (0.11190044845328519+0j), (0.09131649839841018+0j), (0.11246476027166757+0j), (0.09427772585578936+0j), (0.10034007066108216+0j), (0.07918248629720792+0j), (0.09191327237357955+0j), (0.09372064187159158+0j), (0.09785352802640564+0j), (0.10034007066108216+0j), (0.09427772585578936+0j), (0.09191327237357955+0j), (0.07918248629720792+0j), (0.09785352802640564+0j), (0.09372064187159158+0j), (0.11246476027166782+0j), (0.079182486297208+0j), (0.09191327237357963+0j), (0.09372064187159165+0j), (0.0978535280264057+0j), (0.09191327237357963+0j), (0.079182486297208+0j), (0.0978535280264057+0j), (0.09372064187159165+0j), ...]",-8.703164+0.000000j,True
1,Z9,14,"[T_18, T_88, T_169, T_231, T_273, T_309, T_333, T_352, T_360, T_365, T_374, T_375, T_376, T_377]","[(1.1498713328803534+0j), (-0.1422933668714303+0j), (-0.13835155488669257+0j), (-0.09238728853855033+0j), (-0.08001574498509728+0j), (-0.08924796546039968+0j), (-0.08557179384084711+0j), (-0.10034007066108216+0j), (-0.09427772585578936+0j), (-0.11246476027166782+0j), (-0.09191327237357963+0j), (-0.079182486297208+0j), (-0.0978535280264057+0j), (-0.09372064187159165+0j)]",-0.147749+0.000000j,True
2,Z5,14,"[T_13, T_66, T_150, T_216, T_261, T_293, T_325, T_328, T

=== BK Pauli-duplication ratio ===


,mapping,include_identity,sum_alpha_number_of_Pauli_strings,number_of_unique_Pauli_strings_before_cancellation,number_of_Pauli_strings_in_full_H,duplication_ratio,raw_reuse_ratio_before_cancellation
0,BK,True,2201,994,666,3.304805,2.214286



=== Duplicated BK Pauli strings ===


,Pauli_string,number_of_appearances,appears_in_vertices,individual_coefficients,combined_coefficient_in_full_H,survives_in_full_H
0,I,106,"[T_0, T_1, T_4, T_7, T_9, T_11, T_13, T_15, T_16, T_17, T_18, T_19, T_20, T_21, T_22, T_23, T_38, T_44, T_56, T_66, T_74, T_79, T_83, T_88, T_92, T_103, T_108, T_121, T_124, T_136, T_142, T_150, T_157, T_161, T_165, T_169, T_178, T_183, T_192, T_195, T_200, T_210, T_216, T_222, T_225, T_228, T_231, T_234, T_241, T_245, T_253, T_255, T_261, T_265, T_268, T_270, T_273, T_279, T_283, T_288, T_290, T_293, T_300, T_303, T_306, T_309, T_312, T_315, T_318, T_323, T_325, T_328, T_330, T_333, T_335, T_338, T_343, T_345, T_346, T_350, T_352, T_353, T_355, T_356, T_358, T_359, T_360, T_361, T_362, T_363, T_364, T_365, T_368, T_370, T_371, T_373, T_374, T_375, T_376, T_377, ...]","[(3.3921616084615387+0j), (-4.326807030263166+0j), (-4.326807030263166+0j), (-1.2340546110559247+0j), (-1.2340546110559247+0j), (-1.215245368365108+0j), (-1.215245368365108+0j), (-1.1498713328803523+0j), (-1.1498713328803523+0j), (-1.1498713328803534+0j), (-1.1498713328803534+0j), (-0.9578827479006828+0j), (-0.9578827479006828+0j), (-0.8989090819408376+0j), (-0.8989090819408376+0j), (0.5678722535171743+0j), (0.11545479210392827+0j), (0.1221514437504704+0j), (0.11327600938641182+0j), (0.11478797608484514+0j), (0.1383515548866924+0j), (0.14229336687143013+0j), (0.13835155488669257+0j), (0.1422933668714303+0j), (0.11317552161416361+0j), (0.1190723271583569+0j), (0.13917930357851754+0j), (0.14453738718522274+0j), (0.1221514437504704+0j), (0.11545479210392827+0j), (0.11478797608484514+0j), (0.11327600938641182+0j), (0.14229336687143013+0j), (0.1383515548866924+0j), (0.1422933668714303+0j), (0.13835155488669257+0j), (0.1190723271583569+0j), (0.11317552161416361+0j), (0.14453738718522274+0j), (0.13917930357851754+0j), (0.0997579308471877+0j), (0.061966708537915284+0j), (0.10309654502523614+0j), (0.08001574498509717+0j), (0.09238728853855022+0j), (0.08001574498509728+0j), (0.09238728853855033+0j), (0.07994055092939271+0j), (0.09934427228949878+0j), (0.09305187375107359+0j), (0.10719802386527671+0j), (0.10309654502523614+0j), (0.061966708537915284+0j), (0.09238728853855022+0j), (0.08001574498509717+0j), (0.09238728853855033+0j), (0.08001574498509728+0j), (0.09934427228949878+0j), (0.07994055092939271+0j), (0.10719802386527671+0j), (0.09305187375107359+0j), (0.10888573943004314+0j), (0.08557179384084701+0j), (0.08924796546039956+0j), (0.08557179384084711+0j), (0.08924796546039968+0j), (0.08124465742865461+0j), (0.1021021909745089+0j), (0.09131649839841018+0j), (0.11190044845328519+0j), (0.08924796546039956+0j), (0.08557179384084701+0j), (0.08924796546039968+0j), (0.08557179384084711+0j), (0.1021021909745089+0j), (0.08124465742865461+0j), (0.11190044845328519+0j), (0.09131649839841018+0j), (0.11246476027166757+0j), (0.09427772585578936+0j), (0.10034007066108216+0j), (0.07918248629720792+0j), (0.09191327237357955+0j), (0.09372064187159158+0j), (0.09785352802640564+0j), (0.10034007066108216+0j), (0.09427772585578936+0j), (0.09191327237357955+0j), (0.07918248629720792+0j), (0.09785352802640564+0j), (0.09372064187159158+0j), (0.11246476027166782+0j), (0.079182486297208+0j), (0.09191327237357963+0j), (0.09372064187159165+0j), (0.0978535280264057+0j), (0.09191327237357963+0j), (0.079182486297208+0j), (0.0978535280264057+0j), (0.09372064187159165+0j), ...]",-8.703164+0.000000j,True
1,Z8 Z9,14,"[T_18, T_88, T_169, T_231, T_273, T_309, T_333, T_352, T_360, T_365, T_374, T_375, T_376, T_377]","[(1.1498713328803534+0j), (-0.1422933668714303+0j), (-0.13835155488669257+0j), (-0.09238728853855033+0j), (-0.08001574498509728+0j), (-0.08924796546039968+0j), (-0.08557179384084711+0j), (-0.10034007066108216+0j), (-0.09427772585578936+0j), (-0.11246476027166782+0j), (-0.09191327237357963+0j), (-0.079182486297208+0j), (-0.0978535280264057+0j), (-0.09372064187159165+0j)]",-0.147749+0.000000j,True
2,Z4 Z5,14,"[T_13, T_66, T_150, T_216, T_261, T_293, T_325, T_